In [1]:
import joblib
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

In [3]:
# Cargar modelo completo
modelo_clasificador = joblib.load('../binary/model_clasificador_lixiviados.joblib')

scaler = modelo_clasificador['scaler']
pca_model = modelo_clasificador['pca']
col_names = modelo_clasificador['columnas_originales']
cluster_centers = modelo_clasificador['cluster_centers_pca']

In [4]:
def clasificar_muestra(chemical_res, scaler, pca_model, cluster_centers):
    """
    chemical_res: Una lista o array con los 16 valores químicos.
    """
    # 1. Asegurar que los datos tengan la forma correcta (1, 16)
    # muestra = np.array(chemical_res).reshape(1, -1)
    
    # 2. Escalado (Fundamental: usa la media y desviación del entrenamiento)
    muestra_escalada = scaler.transform(chemical_res)
    
    # 3. Reducción PCA (Bajar de 16 a las componentes que definiste)
    muestra_pca = pca_model.transform(muestra_escalada)
    
    # --- AJUSTE DE DIMENSIONES SI ES NECESARIO ---
    # Si tu ABC solo optimizó 10 componentes pero el PCA entrega 15 (como vimos en el error anterior)
    # Tomamos solo las primeras 10 componentes que son las que el ABC conoce
    muestra_pca_reducida = muestra_pca[:, :10] 
    
    # 4. Cálculo de distancias a los 3 centroides
    # 'centroides' tiene forma (3, 10)
    distancias = cdist(muestra_pca_reducida, cluster_centers, metric='euclidean')
    
    # 5. Obtener el cluster más cercano
    cluster_asignado = np.argmin(distancias)
    
    return cluster_asignado, distancias

In [5]:
# Supongamos que tienes una nueva muestra de agua con 16 parámetros:
col_names = ['cadmio','cobre','coliformes_termotolerantes','coliformes_totales','cromo','dbo5',
             'dqo','hierro','manganeso','niquel','ph','plomo','solidos_suspendidos',
             'solidos_totales','zinc']

muestras_quimicas = np.array([
    [0.5, 2.89, 13000.0, 13000.0, 2.0, 0.35, 7.75, 1.093, 0.03675, 5.2, 8.44, 10.0, 0.0, 293.0, 0.1172],
    [0.5, 10.36, 2800.0, 8000.0, 19.7, 0.4, 3.7, 0.262, 0.0706, 50.0, 7.72, 10.0, 121.0, 434.0, 0.032],
    [20.0, 13.4, 16000.0, 16000.0, 50.0, 0.3, 12.0, 2.106, 0.035, 27.7, 8.1, 100.0, 37.0, 455.0, 0.0239],
    [10.52, 143.41, 5400.0, 9200.0, 416.03, 117.23, 7.725, 20.644, 1.458, 373.83, 8.32, 64.2, 120.0, 12.344, 1.252]
])


for i in range(muestras_quimicas.shape[0]):
    muestra_quimica = pd.DataFrame([muestras_quimicas[i]], columns=col_names)
    display(muestra_quimica)

    cluster, dits = clasificar_muestra(
        chemical_res=muestra_quimica, 
        scaler=scaler, 
        pca_model=pca_model, 
        cluster_centers=cluster_centers
    )

    print(f"La muestra ha sido clasificada en el Cluster: {cluster}, distancias: {dits}")

,cadmio,cobre,coliformes_termotolerantes,coliformes_totales,cromo,dbo5,dqo,hierro,manganeso,niquel,ph,plomo,solidos_suspendidos,solidos_totales,zinc
0,0.5,2.89,13000.0,13000.0,2.0,0.35,7.75,1.093,0.03675,5.2,8.44,10.0,0.0,293.0,0.1172


La muestra ha sido clasificada en el Cluster: 3, distancias: [[ 3.80990486 20.96195404  9.06384616  1.33922147]]


,cadmio,cobre,coliformes_termotolerantes,coliformes_totales,cromo,dbo5,dqo,hierro,manganeso,niquel,ph,plomo,solidos_suspendidos,solidos_totales,zinc
0,0.5,10.36,2800.0,8000.0,19.7,0.4,3.7,0.262,0.0706,50.0,7.72,10.0,121.0,434.0,0.032


La muestra ha sido clasificada en el Cluster: 3, distancias: [[ 3.36979524 20.83385408  8.50161637  1.11193252]]


,cadmio,cobre,coliformes_termotolerantes,coliformes_totales,cromo,dbo5,dqo,hierro,manganeso,niquel,ph,plomo,solidos_suspendidos,solidos_totales,zinc
0,20.0,13.4,16000.0,16000.0,50.0,0.3,12.0,2.106,0.035,27.7,8.1,100.0,37.0,455.0,0.0239


La muestra ha sido clasificada en el Cluster: 3, distancias: [[ 3.25746122 20.82971073  7.89689753  0.9859053 ]]


,cadmio,cobre,coliformes_termotolerantes,coliformes_totales,cromo,dbo5,dqo,hierro,manganeso,niquel,ph,plomo,solidos_suspendidos,solidos_totales,zinc
0,10.52,143.41,5400.0,9200.0,416.03,117.23,7.725,20.644,1.458,373.83,8.32,64.2,120.0,12.344,1.252


La muestra ha sido clasificada en el Cluster: 0, distancias: [[ 1.59862844 20.93741169  7.68993713  3.22119439]]
